# LangChain Chains

## What is a Chain?
A **Chain** connects multiple components (Prompt → LLM → Output Parser) into a single workflow.

```text
Input → Prompt → LLM → Output Parser → Result
```

---

## Why use Chains?
- Reusable workflows
- Cleaner code
- Easy to combine multiple LangChain components
- Foundation for building RAGs and Agents

---

## Basic Chain

```python
chain = prompt | llm
response = chain.invoke({"topic": "Transformers"})
```

---

## Chain Components

- **PromptTemplate** → Formats the input
- **LLM / ChatModel** → Generates response
- **OutputParser** → Converts output to desired format

```text
Prompt → LLM → Parser
```

---

## LCEL (LangChain Expression Language)

Uses the `|` operator to connect components.

```python
chain = prompt | llm | parser
```

---

## Invoke a Chain

Single input:

```python
chain.invoke({"topic": "AI"})
```

Batch input:

```python
chain.batch([
    {"topic": "AI"},
    {"topic": "ML"}
])
```

Stream output:

```python
for chunk in chain.stream({"topic": "AI"}):
    print(chunk)
```

---

## Summary

- Chain = Workflow
- Connects Prompt → LLM → Parser
- Uses `|` operator (LCEL)
- Main methods:
  - `invoke()`
  - `batch()`
  - `stream()`
- Makes LangChain applications modular and reusable.

## 1) Simple Sequential Chain

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.7
)

prompt = PromptTemplate(
    template="Generate 5 interesting facts about {topic}",
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template="Summarize it. {text}",
    input_variables=['text']
)

parser = StrOutputParser()

In [9]:
chain = prompt | model | parser | prompt2 | model | parser

response = chain.invoke({'topic' : 'transformer NN'})

print(response)

Transformer neural networks, introduced in 2017 by Vaswani et al., revolutionized natural language processing by using self-attention mechanisms instead of recurrent or convolutional layers. They enable efficient parallel processing of sequences, allowing faster training and handling longer inputs. Transformers scale well with data and model size, underpinning large language models like GPT and BERT. Their multi-head attention mechanism captures diverse contextual information simultaneously, enhancing understanding of complex language tasks. Originally for NLP, Transformer architectures have since been successfully adapted to fields like computer vision, speech, and reinforcement learning, showcasing their broad versatility.


## 2) Parallel Chain

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.7
)

parser = StrOutputParser()

In [12]:
prompt1 = PromptTemplate(
    template="Generate 5 interesting facts about {text}",
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template="Summarize the text: {text}",
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template="Combine both {points} and {summary}",
    input_variables=['points', 'summary']
)

In [13]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel({
    "points" : prompt1 | model | parser,
    "summary" : prompt2 | model| parser
})


chain = parallel_chain | prompt3 | model | parser

In [14]:
text = """
Transformers acts as the model-definition framework for state-of-the-art machine learning models in text, computer vision, audio, video, and multimodal model, for both inference and training.

It centralizes the model definition so that this definition is agreed upon across the ecosystem. transformers is the pivot across frameworks: if a model definition is supported, it will be compatible with the majority of training frameworks (Axolotl, Unsloth, DeepSpeed, FSDP, PyTorch-Lightning, …), inference engines (vLLM, SGLang, TGI, …), and adjacent modeling libraries (llama.cpp, mlx, …) which leverage the model definition from transformers.

We pledge to help support new state-of-the-art models and democratize their usage by having their model definition be simple, customizable, and efficient.
"""

response = chain.invoke({'text' : text})

print(response)

Here is a combined and cohesive summary incorporating both inputs:

---

**Transformers: A Unified Model-Definition Framework**

Transformers is a unified, extensible framework designed for defining state-of-the-art machine learning models across multiple domains—including text, computer vision, audio, video, and multimodal applications. It standardizes model definitions to ensure consistency and compatibility across a wide range of training frameworks (such as Axolotl, Unsloth, DeepSpeed, FSDP, PyTorch-Lightning) and inference engines (including vLLM, SGLang, TGI), simplifying both model development and deployment.

By serving as a universal model-definition standard, Transformers acts as a pivot that facilitates interoperability not only between training and inference tools but also across numerous adjacent modeling libraries like llama.cpp and mlx. This shared foundation enables developers and researchers to build on and extend existing architectures without needing to reinvent them

### 3) Conditional Chain

In [61]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = PromptTemplate(
    template="""
Analyze the sentiment of the following review.

Review:
{text}

Return only a JSON object.

{format_instructions}

Rules:
- sentiment must be exactly "positive" or "negative".
- Return second value as Review itself.
""",
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

prompt2 = PromptTemplate(
    template="Summarize movie in short paragraph, take movie name from text. {review}",
    input_variables=['review']
)

prompt3 = PromptTemplate(
    template="Summarize movie in short points, take movie name from text.. {review}",
    input_variables=['review']
)

parser2 = StrOutputParser()

In [62]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

conditional_chain = RunnableBranch(
    (lambda x:'positive' in x['sentiment'].strip().lower(), prompt2 | model | parser2),
    (lambda x:'negative' in x['sentiment'].strip().lower(), prompt3 | model | parser2),
    RunnableLambda(lambda x: "Invalid sentiment")
)

In [65]:
sentiment_chain = prompt | model | parser

chain = sentiment_chain | conditional_chain

response = chain.invoke({'text' : text})

print(response)

- Movie: Transformer  
- Overall impression: Okay  
- Length: Very short  
- Expectation: Hope for something more or extended version in the future
